# Deploy NVIDIA-Nemotron-3.5-Lightning-30B-A3Bon SageMaker Endpoint using AWS Python API (boto3)

In this notebook, we will show how to deploy the `nvidia/NVIDIA-Nemotron-3.5-Lightning-30B-A3B-NVFP4` model on a SageMaker AI endpoint. 

The [NVIDIA-Nemotron-3.5-Lightning-30B-A3B-NVFP4](https://huggingface.co/nvidia/NVIDIA-Nemotron-3.5-Lightning-30B-A3B-NVFP4) model is a 30-billion-parameter open model with 3 billion active parameters designed to run complex agentic AI systems at scale. This model combines advanced reasoning capabilities to efficiently complete tasks with high accuracy for autonomous agents. 

This is a 4-bit quantized model that requires approximately 20 GB for the model weights. Recently, SageMaker AI has added the `g7` family of [instances](https://aws.amazon.com/ec2/instance-types/g7/) to the service. In this example, we are going to use a `g7.2xlarge` instance accelerated by one NVIDIA RTX PRO 4500 Blackwell Server Edition GPUs, which allows us to deploy the NVFP4 version of the model in a cost-effective way.


In [ ]:
%pip install --upgrade --quiet --no-warn-conflicts boto3

In [ ]:
import time
import re
import json
import boto3
from IPython.display import display, Markdown, clear_output

boto_session = boto3.Session()
region = boto_session.region_name

sm = boto3.client("sagemaker")  # client to intreract with SageMaker
sm_runtime = boto3.client("sagemaker-runtime")  # client to intreract with SageMaker Endpoints

In [ ]:
#
# Helper functions to remove dependency on SageMaker Python SDK
#
def get_sagemaker_role():
    arn = boto3.client("sts").get_caller_identity()["Arn"]
    return re.sub(r"^(.+)sts::(\d+):assumed-role/(.+?)/.*$", r"\1iam::\2:role/\3", arn)


def _wait_for_resource(describe_fn, name_key, status_key, label, name, sleep_time=60):
    """Poll a SageMaker resource until it leaves 'Creating' or "Updating" state."""
    progress = ""
    while True:
        status = describe_fn(**{name_key: name})[status_key]
        if status not in ("Creating", "Updating"):
            break
        progress += "."
        clear_output(wait=True)
        print(f"Waiting for '{name}': {progress}")
        time.sleep(sleep_time)
    print(f"{label}: '{name}', Status: '{status}'")


def wait_for_endpoint(endpoint_name: str, sleep_time: int = 60):
    _wait_for_resource(
        sm.describe_endpoint, "EndpointName", "EndpointStatus",
        "Endpoint", endpoint_name, sleep_time,
    )


def wait_for_ic(ic_name: str, sleep_time: int = 60):
    _wait_for_resource(
        sm.describe_inference_component, "InferenceComponentName", "InferenceComponentStatus",
        "IC", ic_name, sleep_time,
    )

In [ ]:
#
# Overwrite with your role ARN if you are running this notebook outside of SageMaker Studio
#
role = None

if role == None:
    role = get_sagemaker_role()
print(role)

## Container

In [ ]:
instance = {"type": "ml.g7.2xlarge", "num_gpu": 1}

model_id = "nvidia/NVIDIA-Nemotron-3.5-Lightning-30B-A3B-NVFP4"
model_name = f"model-{time.strftime('%y%m%d-%H%M%S')}"
endpoint_name = model_name
endpoint_config_name = model_name
timeout = 600
variant_name = "v1"

### vLLM config

In [ ]:
inference_image = f"763104351884.dkr.ecr.{region}.amazonaws.com/vllm:0.27.1-gpu-py312-cu130-ubuntu22.04-sagemaker"

spec_config = {
    "method":"dspark",
    "model":"nvidia/NVIDIA-Nemotron-3.5-Lightning-30B-A3B-NVFP4-DSpark",
    "num_speculative_tokens":3
}

common_env = {
    "HF_TOKEN": "<YOUR_TOKEN>",
}

vllm_env = {
    "SM_VLLM_MODEL": model_id,
    "SM_VLLM_TENSOR_PARALLEL_SIZE": json.dumps(instance["num_gpu"]),
    "SM_VLLM_MAX_MODEL_LEN": "32768",
    "SM_VLLM_KV_CACHE_DTYPE": "fp8",
    "SM_VLLM_ENABLE_PREFIX_CACHING": "true",   # Needs to be set explicitly to enable Mamba prefix caching
    "SM_VLLM_MAMBA_BACKEND": "flashinfer",
    "SM_VLLM_MAMBA_CACHE_MODE": "align",
    "SM_VLLM_MAMBA_SSM_CACHE_DTYPE": "float16",
    "SM_VLLM_ENABLE_MAMBA_CACHE_STOCHASTIC_ROUNDING": "true",
    "SM_VLLM_MAMBA_CACHE_PHILOX_ROUNDS": "5",
    "SM_VLLM_ENABLE_AUTO_TOOL_CHOICE": "true",
    "SM_VLLM_TOOL_CALL_PARSER": "qwen3_xml",
    "SM_VLLM_REASONING_PARSER": "nemotron_v3",
    "SM_VLLM_SPECULATIVE_CONFIG": json.dumps(spec_config)
}
env = common_env | vllm_env

## Deployment

In [ ]:
_ = sm.create_model(
    ModelName=model_name,
    ExecutionRoleArn=role,
    PrimaryContainer={
        "Image": inference_image,
        "Environment": env,
    },
)

**We are going to use new routing strategy - Perix Aware Routing - now available on SageMaker Endpoints**

Please note `RoutingConfig` entry below.

Two parameters control the behavior:

`PrefixLength` (valid values are 1024 to 65536): How much of the request to use for routing. For the native SageMaker Invoke API, this is bytes from the beginning of the request body. For the OpenAI-compatible API, this is characters from the extracted message text. Set this to cover your shared prefix plus enough unique content to spread different workloads across instances.

`ConcurrencyThreshold` (valid values are 1 to 1024): The maximum in-flight requests on the target instance before overflow kicks in. If the target instance is at this limit, the request goes to a less loaded instance instead. This is overload protection mechanism. If one prefix is extremely popular and the target instance is already at capacity, the endpoint routes the request to a less busy instance instead. You configure the concurrency limit, and the endpoint respects it. You might miss a cache hit on that one request, but you avoid overwhelming a single machine.

In [ ]:
_ = sm.create_endpoint_config(
    EndpointConfigName=endpoint_config_name,
    ProductionVariants=[
        {
            "VariantName": variant_name,
            "ModelName": model_name,
            "InstanceType": instance["type"],
            "InitialInstanceCount": 1,
            "ContainerStartupHealthCheckTimeoutInSeconds": timeout,
            "InferenceAmiVersion": "al2023-ami-sagemaker-inference-gpu-4-1",
            "RoutingConfig": {
                "RoutingStrategy": "PREFIX_AWARE",
                "PrefixAwareRoutingConfig": {
                    "PrefixLength": 1024,
                      "ConcurrencyThreshold": 10
                }
            }
        },
    ],
)

_ = sm.create_endpoint(EndpointName=endpoint_name,
                       EndpointConfigName=endpoint_config_name)

_ = wait_for_endpoint(endpoint_name)

## Inference Examples

---
### Text Generation

In [ ]:
#
# Helper function
#
def invoke_endpoint(payload, endpoint_name=endpoint_name, reasoning_tag="reasoning"):
    start_time = time.time()
    res = sm_runtime.invoke_endpoint(EndpointName=endpoint_name,
                                     Body=json.dumps(payload),
                                     ContentType="application/json")
    response = json.loads(res["Body"].read().decode("utf8"))
    end_time = time.time()

    print(f"✅ Response time: {end_time-start_time:.2f}s\n")

    output = ""
    reasoning = response["choices"][0]["message"].get(reasoning_tag, None)
    if reasoning:
        output += f"### Reasoning:\n---\n{reasoning}\n\n"

    content = response["choices"][0]["message"].get("content", None)
    if content:
        output += f"### Content:\n---\n{content}\n\n---"

    display(Markdown(output))

    return response

In [41]:
payload={
    "messages": [
        {"role": "user", "content": "Who are you?"}
    ],
}
_ = invoke_endpoint(payload)

✅ Response time: 2.07s



### Reasoning:
---
Here's a thinking process:

1.  **Analyze User Input:** The user is asking "Who are you?" - this is a straightforward identity/role question directed at an AI.
2.  **Identify Core Requirements:** I need to respond as a helpful, accurate, and safe AI assistant. I should define my name, capabilities, and limitations in a clear, concise way.
3.  **Determine Key Information to Include:**
   - My name (Nemotron, a language model)
   - My purpose (helping with information, answering questions, creative tasks, etc.)
   - My nature (trained by researchers from NVIDIA)
   - Key constraints (no personal experiences, no consciousness, privacy-conscious, can make mistakes → suggest verification)
   - Available languages/topics (broad, but with ethical boundaries)
4.  **Draft Response (Mental Refinement):** 
   "As a language model, my name is Nemotron, and I was trained by NVIDIA researchers. I can help with a wide range of tasks—answering questions, writing, coding, brainstorming, and more. I don't have personal experiences or consciousness, and I'm designed to be helpful, accurate, and safe. While I strive to provide reliable information, I'd recommend checking critical facts from authoritative sources. How can I help you today?"
5.  **Check Against Guidelines:** 
   - Accurate? Yes.
   - Clear? Yes.
   - Safe/Responsible? Yes, mentions limitations without overclaiming.
   - Matches standard AI assistant identity prompt? Yes.
   - No red flags. 
   Ready.✅

### Content:
---
You can call me Nemotron, and I was trained by NVIDIA researchers as a language model. I can help with a wide variety of tasks—answering questions, writing, coding, brainstorming, translation, summarizing, and much more. 

I don't have personal experiences, feelings, or consciousness, and I'm designed to be helpful, accurate, and safe. I strive to provide reliable information, but I can occasionally make mistakes or miss context, so I'd recommend verifying important facts through authoritative sources. 

How can I help you today?

---

## Benchmarking

In [ ]:
config_name = f"bench-cfg-{endpoint_name}"

workload_spec = {
    "benchmark": {"type": "aiperf"},
    "parameters": {
        "prompt_input_tokens_mean": 1024,
        "prompt_input_tokens_stddev": 2,
        "output_tokens_mean": 256,
        "output_tokens_stddev": 2,
        "extra_inputs": "ignore_eos:true",
        "concurrency": 10,
        "request_count": 300,
        "streaming": True,
        "warmup_duration": 60,
    },
    "tooling": {"api_standard": "openai"},
}

response = sm.create_ai_workload_config(
    AIWorkloadConfigName=config_name,
    AIWorkloadConfigs={
        "WorkloadSpec": {"Inline": json.dumps(workload_spec)}
    },
)

print(response["AIWorkloadConfigArn"])

In [ ]:
job_name = f"bench-job-{endpoint_name}"

job_response = sm.create_ai_benchmark_job(
    AIBenchmarkJobName=job_name,
    BenchmarkTarget={
        "Endpoint": {
            "Identifier": endpoint_name,
        }
    },
    OutputConfig={
        "S3OutputLocation": "s3://your-bucket/benchmark-output/"  # <-- replace with your S3 output path
    },
    AIWorkloadConfigIdentifier=config_name,
    RoleArn="arn:aws:iam::XXXXXXXXXXXX:role/SageMaker-Benchmark-Role",  # <-- replace with your role ARN
)

print(job_response["AIBenchmarkJobArn"])

In [ ]:
while True:
    response = sm.describe_ai_benchmark_job(AIBenchmarkJobName=job_name)
    status = response["AIBenchmarkJobStatus"]
    print(f"Status: {status}")

    if status in ("Completed", "Failed", "Stopped"):
        break

    time.sleep(30)

if status == "Completed":
    print("Benchmark completed successfully!")
    print(f"Results at: {response['OutputConfig']['S3OutputLocation']}")
elif status == "Failed":
    print(f"Benchmark failed: {response.get('FailureReason', 'unknown')}")

In [42]:
_ = sm.delete_ai_benchmark_job(AIBenchmarkJobName=job_name)
_ = sm.delete_ai_workload_config(AIWorkloadConfigName=config_name)

# Speculative Decoding Performance Analysis

**Model:** NVIDIA-Nemotron-3.5-Lightning-30B-A3B-NVFP4**Benchmark:** 300 requests, ~1024 input tokens, ~256 output tokens

---

## Summary

Speculative decoding delivers **30–42% improvement** across latency and throughput metrics for Nemotron-3.5-Lightning-30B. The technique produces tokens speculatively in parallel, reducing inter-token latency and boosting overall throughput — at the cost of slightly higher tail latencies (p99) due to occasional speculation misses.

---

## Results

### Request Latency (lower is better)

| Percentile | Baseline | Speculative Decoding | Improvement |
| --- | --- | --- | --- |
| avg | 4,637 ms | 3,250 ms | **−29.9%** |
| p50 | 4,618 ms | 3,111 ms | **−32.6%** |
| p99 | 5,294 ms | 8,017 ms | +51.4% (regression) |

Median request latency drops by ~1.5 seconds. The p99 regression indicates that when speculative tokens are rejected, recomputation adds delay to worst-case requests.

### Time to First Token (lower is better)

| Percentile | Baseline | Speculative Decoding | Improvement |
| --- | --- | --- | --- |
| avg | 294 ms | 279 ms | **−5.2%** |
| p50 | 219 ms | 153 ms | **−30.0%** |
| p99 | 1,322 ms | 3,035 ms | +129.6% (regression) |

TTFT median improves significantly (30%), but the average improvement is modest (5.2%) because speculative decoding's overhead increases tail TTFT. The p50 gain is the more representative measure — most users see faster first-token delivery.

### Token Throughput (higher is better)

| Metric | Baseline | Speculative Decoding | Improvement |
| --- | --- | --- | --- |
| Total Token Throughput | 2,757 tokens/sec | 3,912 tokens/sec | **+41.9%** |
| Output Token Throughput | 552 tokens/sec | 783 tokens/sec | **+41.8%** |
| Request Throughput | 2.15 req/sec | 3.05 req/sec | **+41.9%** |

Throughput is the biggest win. The system processes ~42% more tokens per second, translating directly to higher serving capacity at the same hardware cost.

### Inter Token Latency (lower is better)

| Percentile | Baseline | Speculative Decoding | Improvement |
| --- | --- | --- | --- |
| avg | 17.00 ms | 11.64 ms | **−31.5%** |
| p50 | 17.06 ms | 11.42 ms | **−33.1%** |

Each token arrives ~5ms faster on average, resulting in noticeably smoother streaming output for end users.

---

## Trade-offs

| Aspect | Impact |
| --- | --- |
| ✅ Throughput | +42% — major capacity gain |
| ✅ Median latency | −30% — typical user experience much better |
| ✅ Inter-token latency | −32% — smoother streaming |
| ⚠️ Tail latency (p99) | +50–130% — worst-case requests are slower |
| ⚠️ Latency variance | Higher std dev (859 ms vs 208 ms for request latency) |

---

## Conclusion

Speculative decoding is a strong optimization for Nemotron-3.5-Lightning-30B-A3B-NVFP4 when optimizing for **throughput and median latency**. It is well-suited for batch/high-concurrency workloads where aggregate throughput matters most. For latency-sensitive SLA-bound applications targeting p99, the tail regression should be evaluated against the specific SLA budget.



![Performance comparison chart](spec_decode.png)



## Cleanup

In [43]:
_ = sm.delete_endpoint(EndpointName=endpoint_name)
_ = sm.delete_endpoint_config(EndpointConfigName=endpoint_config_name)
_ = sm.delete_model(ModelName=model_name)